# Solution P2 · House prices: from the prediction to trust in the prediction

**Type**: Implement with scikit-learn

## Objective

Build a regression model with XGBoost on California housing and then answer two questions a customer would really ask: **how reliable** is the number you estimate if the houses to predict are in an area never seen (spatial validation) and **with what margin** (90% prediction intervals with quantile regression and their coverage).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold, KFold, cross_val_score, train_test_split

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
CITIES = {"LA": (34.05, -118.24), "SF": (37.77, -122.42), "SD": (32.72, -117.16)}


def new_xgb():
    return xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=0)


def quantile_model(alpha):
    return GradientBoostingRegressor(loss="quantile", alpha=alpha, n_estimators=150, max_depth=4, learning_rate=0.1, subsample=0.8, random_state=0)

## Commented solution

In [ ]:
def enrich(X):
    F = X.copy()
    F["rooms_per_person"] = F["AveRooms"] / F["AveOccup"]
    F["bedroom_share"] = F["AveBedrms"] / F["AveRooms"]
    for name, (lat, lon) in CITIES.items():
        F[f"d_{name}"] = np.hypot(F["Latitude"] - lat, (F["Longitude"] - lon) * 0.8)
    F["d_city"] = F[[f"d_{n}" for n in CITIES]].min(axis=1)
    return F


rmse = lambda a, b: mean_squared_error(a, b, squared=False)
base_rmse = rmse(y_te, new_xgb().fit(X_tr, y_tr).predict(X_te))
fe_rmse = rmse(y_te, new_xgb().fit(enrich(X_tr), y_tr).predict(enrich(X_te)))
print(f"test RMSE: original features {base_rmse:.4f} | with the new features {fe_rmse:.4f} ({1 - fe_rmse / base_rmse:.1%} less)")

**Spatial validation.** In random cross-validation a district has, in the training set, almost identical neighbors (same neighborhood, same prices). With `GroupKFold` on the **geographic blocks** the districts of the validation fold are in areas the model has never seen: it is the scenario of someone who uses the model for a new city.

In [ ]:
Fe = enrich(X_tr)
blocks = (np.floor(X_tr["Latitude"]).astype(int).astype(str) + "_" + np.floor(X_tr["Longitude"]).astype(int).astype(str)).to_numpy()
n_blocks = len(set(blocks))
random_rmse = float(-cross_val_score(new_xgb(), Fe, y_tr, cv=KFold(5, shuffle=True, random_state=0), scoring="neg_root_mean_squared_error").mean())
spatial_fold_rmse = (-cross_val_score(new_xgb(), Fe, y_tr, cv=GroupKFold(5), groups=blocks, scoring="neg_root_mean_squared_error")).tolist()
spatial_rmse = float(np.mean(spatial_fold_rmse))
overestimate = spatial_rmse / random_rmse - 1
print(f"{n_blocks} blocks | RMSE with random CV {random_rmse:.3f} | with spatial CV {spatial_rmse:.3f} ({overestimate:+.0%}) | per fold: {np.round(spatial_fold_rmse, 3)}")

**Intervals with quantile regression.** A model with `loss="quantile"` and `alpha=0.05` estimates the 5% quantile of the price given `X`; with `alpha=0.95` the 95% quantile. The interval `[q05, q95]` should contain the true value in 90% of the cases: we check it.

In [ ]:
idx = np.random.default_rng(0).permutation(len(X_tr))[:10000]
Fte = enrich(X_te)
q05 = quantile_model(0.05).fit(Fe.iloc[idx], y_tr.iloc[idx])
q95 = quantile_model(0.95).fit(Fe.iloc[idx], y_tr.iloc[idx])
low, high = q05.predict(Fte), q95.predict(Fte)
test_coverage = float(((y_te >= low) & (y_te <= high)).mean())
mean_width = float((high - low).mean())
print(f"coverage on the test set (nominal 0.90): {test_coverage:.3f} | mean width of the interval: {mean_width:.2f} (that is {mean_width * 100:.0f} thousand dollars)")

In [ ]:
spatial_coverages = []
for a, b in GroupKFold(5).split(Fe, y_tr, blocks):
    sub = a[np.random.default_rng(0).permutation(len(a))[:10000]]
    lo = quantile_model(0.05).fit(Fe.iloc[sub], y_tr.iloc[sub])
    hi = quantile_model(0.95).fit(Fe.iloc[sub], y_tr.iloc[sub])
    yv = y_tr.iloc[b]
    spatial_coverages.append(float(((yv >= lo.predict(Fe.iloc[b])) & (yv <= hi.predict(Fe.iloc[b]))).mean()))
mean_spatial_coverage, min_coverage = float(np.mean(spatial_coverages)), float(min(spatial_coverages))
print(f"coverage per spatial fold: {np.round(spatial_coverages, 3)} | mean {mean_spatial_coverage:.3f} | minimum {min_coverage:.3f}")

In [ ]:
report = {
    "test_rmse": float(fe_rmse), "random_rmse": random_rmse, "spatial_rmse": spatial_rmse,
    "test_coverage": test_coverage, "mean_spatial_coverage": mean_spatial_coverage,
    "message": f"In an area never seen the typical error is about {spatial_rmse * 100:.0f} thousand dollars (not {random_rmse * 100:.0f}) and the 90% interval covers on average only {mean_spatial_coverage:.0%} of the districts.",
}
report

## Check

In [ ]:
with verify():
    Fe = enrich(X_tr)
    check({"rooms_per_person", "bedroom_share", "d_LA", "d_SF", "d_SD", "d_city"} <= set(Fe.columns) and len(Fe) == len(X_tr) and np.isfinite(Fe.to_numpy()).all() and (Fe["d_city"] <= Fe[["d_LA", "d_SF", "d_SD"]].max(axis=1)).all(), "six new columns, with no NaN or infinities, d_city is the minimum", "np.hypot(Latitude - lat, (Longitude - lon) * 0.8)")
    check(base_rmse is not None and fe_rmse < base_rmse - 0.008 and fe_rmse < 0.44, "the new features reduce the test RMSE by over 0.008 (below 0.44)", "XGBoost on enrich(X_tr), test on enrich(X_te)")
    check(blocks is not None and len(blocks) == len(X_tr) and 40 <= n_blocks <= 70, "between 40 and 70 blocks of 1 degree by 1 degree", "np.floor(Latitude), np.floor(Longitude)")
    check(0.42 < random_rmse < 0.48 and len(spatial_fold_rmse) == 5 and close(spatial_rmse, np.mean(spatial_fold_rmse)), "RMSE with random CV between 0.42 and 0.48; five spatial folds", "scoring='neg_root_mean_squared_error' (flip the sign)")
    check(overestimate > 0.25 and max(spatial_fold_rmse) > 1.4 * min(spatial_fold_rmse), "the spatial error exceeds the random one by over 25% and the spatial folds differ by over 40% among themselves", "spatial_rmse / random_rmse - 1")
    check(test_coverage is not None and 0.80 < test_coverage < 0.93 and mean_width > 1.0, "coverage on the test set between 0.80 and 0.93 (nominal 0.90); mean width over 1.0 (100 thousand dollars)", "(y_te >= q05) & (y_te <= q95)")
    check(len(spatial_coverages) == 5 and mean_spatial_coverage < test_coverage - 0.03 and min_coverage < 0.8, "in new areas the mean coverage drops by over 3 points below that of the test set and in the worst fold it is below 0.80", "GroupKFold(5) with the blocks")
    check(set(report) >= {"test_rmse", "random_rmse", "spatial_rmse", "test_coverage", "mean_spatial_coverage", "message"} and isinstance(report["message"], str) and len(report["message"]) > 40, "complete report", "a dictionary with the required keys")
done()

## What you learned

- **Geographic feature engineering pays:** distances from three big cities and rooms per person take the test RMSE from 0.446 to 0.428 (−4%): a modest gain but almost free, because you add information that trees struggle to build on their own (a distance is a combination of two coordinates).
- **Random validation underestimates the error in new areas by 40%:** RMSE 0.447 with `KFold` against 0.625 with `GroupKFold` on the geographic blocks. Neighboring districts look alike and the random partition puts them in both the training and the validation set. If the model will be used for a new area, the number to report is the second.
- **The risk depends on the area:** the five spatial folds give RMSE from 0.48 to 0.81: the estimate with a single validation area can be very different from the mean. That is why you report the mean and the dispersion of the folds, not a single number.
- **An interval has to be verified, not only built:** the 5%-95% quantiles cover 86% of the cases on the test set (nominal 90%, a bit narrow) and only about 80% in new areas, with one fold at 75%. The "declared" uncertainty is that of the data the model has seen; in new areas a more cautious calibration is needed (wider intervals, calibration per area).

## Common pitfall

Reporting the error of a random cross-validation for a model that will be used in new areas (or periods, or on new people): the validation protocol must look like the real use.

## Interview question

<details><summary>How would you estimate the error of a price model to be used in a city not present in the training set?</summary>

With a geographic block validation (GroupKFold on cells or cities): I keep entire areas out and measure the error on those. The random partition is optimistic because neighbors look alike. I report the mean and the dispersion across the areas, and I check that the prediction intervals keep the declared coverage outside the area too.

</details>